# പാഠം 13 - ഏജന്റ് മെമ്മറി


## സജ്ജീകരണം

ഈ നോട്ട്‌ബുക്ക് **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് **സ്ഥിരമായ മെമ്മറി** ഉള്ള ഒരു യാത്ര ബുക്കിംഗ് ഏജന്റ് എങ്ങനെ നിർമ്മിക്കാമെന്ന് കാണിക്കുന്നു.

ഏജന്റിന്റെ മെമ്മറി വിവിധ തരം — പ്രവര്‍ത്തന, ചെറുകാല, ദീർഘകാല — സംഭാഷണങ്ങളിലൂടെയുള്ള വിവരസംഗ്രഹത്തിലും ഉപയോഗത്തിലും എങ്ങനെ സ്വാധീനിക്കുന്നു എന്ന് നിങ്ങൾ പഠിക്കും.

**ആവശ്യമായ മുൻപരിചയം:**
- ഒരു Microsoft Foundry പ്രോജക്റ്റിൽ വിന്യസിച്ച ചാറ്റ് മോഡൽ (ഉദാഹരണം: `gpt-5-mini`).
- Azure CLI-യിൽ ലോഗിൻ ചെയ്തിരിക്കണം — ടർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
- `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്റ്റിന്റെ എൻഡ്‌പോയിന്റ്.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങളുടെ വിന്യസിച്ച മോഡലിന്റെ പേര്.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ഏജന്റ് ഓർമയുടെ തരം

AI ഏജന്റുകൾ വ്യത്യസ്ത ലക്ഷ്യങ്ങൾക്കായി സേവനം ചെയ്യുന്ന വ്യത്യസ്ത തരത്തിലുള്ള ഓർമ്മകൾ ഉപയോഗിക്കാം:

### പ്രവർത്തന ഓർമ്മ
സംഭാഷണ ത്രെഡ് തന്നെ — ഒരു സെഷനിൽ കൈമാറിയ സന്ദേശങ്ങൾ. ഏജന്റ് സeusρυക്ഷമത നിലനിർത്താൻ ഒരേ ത്രെഡിലുള്ള മുമ്പത്തെ സന്ദേശങ്ങളെ പരാമർശിക്കാം. MAF-ൽ ഇത് **`agent.create_session()`** ഉപയോഗിച്ച് സൃഷ്ടിക്കപ്പെടുന്നു, ഇത് ഒരു `AgentSession` നൽകുന്നു.

### ചുരുക്കകാല ഓർമ്മ
ഒരു ടാസ്‌ക്ക് അല്ലെങ്കിൽ സെഷൻ തുടർന്നുള്ള സമയത്ത് നിലനിൽക്കുന്ന വിവരങ്ങൾ, എന്നാൽ സ്ഥിരമായി സൂക്ഷിക്കപ്പെടാറില്ല. ഉദാഹരണത്തിന്, ഏജന്റ് ഒന്നിലധികം ടേൺ പ്ലാനിംഗ് സംഭാഷണത്തിൽ duringസത്യം സമാഹരിക്കുകയും അന്തിമ യാത്രാമാലിക നിർമ്മിക്കാൻ അവ ഉപയോഗിക്കയും ചെയ്യാം.

### ദൈർഘ്യകാല ഓർമ്മ
**സെഷനുകൾക്കു മുകളിൽ** നിലനിൽക്കുന്ന ഇഷ്ടാനfimകളും സത്യംകഴിവും. തിരികെ വരുന്ന ഉപയോക്താവ് അവരുടെ ഭക്ഷണപരിധികളും യാത്രാ ശൈലിയുമാറ്റി ആവർത്തിക്കേണ്ടതില്ല. ദൈർഘ്യകാല ഓർമ്മ സാധാരണയായി ഒരു ബാഹ്യ സംഭരണശാലയാൽ പിന്തുണയ്ക്കുന്നു — ഒരു ഡാറ്റാബേസ്, ഫയൽ, അല്ലെങ്കിൽ വെക്ടർ ഇൻഡക്സ് — കൂടാതെ ഉപകരണങ്ങളിലൂടെ ഏജന്റിനുമുമ്പിലേക്ക് എത്തിക്കുന്നു.


## സെഷനുകളുമായി പ്രവർത്തിക്കുന്ന മെമ്മറി

മെമ്മറിയുടെ ഏറ്റവും സിംപിളായ രൂപം സംഭാഷണ സെഷനാണ്. ഒരേ സെഷൻ (`agent.create_session()` വഴി സൃഷ്ടിച്ചത്) തുടർച്ചയായി `agent.run()` കോളുകൾക്ക് നൽകുമ്പോൾ, ഏജന്റിന് ആ സംഭാഷണത്തിന്റെ മുഴുവൻ ചരിത്രവും കാണാം, മുമ്പത്തെ വിവരങ്ങൾ ഓർമ്മിക്കാൻ കഴിയും.

ഒരു യാത്ര ഏജന്റ് സൃഷ്ടിച്ച് പ്രവർത്തിക്കുന്ന മെമ്മറി പ്രദർശിപ്പിക്കാം.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ഏജന്റ് ബജറ്റ് ശരിയായി ഓർമ്മിച്ചിരുന്നത് രണ്ടു സന്ദേശങ്ങളും ഒരേ സെഷനിലാണ് ഉണ്ടായത് കൊണ്ടാണ്. ഇതാണ് **വർക്കിംഗ് മെമ്മറി** — ഇത് സെഷന്റെ ആയുസിനുള്ളിൽ മാത്രമേ നിലനിൽക്കൂ.

### ഒരു പുതിയ ത്രെഡോടെ എന്താണ് സംഭവിക്കുന്നത്?

നാം ഒരു **പുതിയ** സെഷൻ സൃഷ്ടിച്ചാൽ, ഏജന്റിന് മുൻസമ്യോദായത്തിന്റെ ഓർമ്മയില്ല:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ദീർഘകാല സ്മൃതി മാതൃക

ഉപയോക്തൃ മുൻഗണനകൾ **സെഷനുകൾക്ക് ഇടയിൽ** ഓർത്തുവെക്കാൻ, സംഭാഷണ ത്രെഡിനു പുറത്തുള്ള ഒരു സ്ഥിരമായ സംഭരണശേഷി ആവശ്യമാണ്. ഏജന്റ് ഈ സംഭരണശേഷിയിലേക്ക് **ടൂൾസ്** മുഖാന്തിരം പ്രവേശിക്കുന്നു — വിവരങ്ങൾ സംരക്ഷിക്കാനും തിരികെ നേടാനും വിളിക്കാൻ കഴിയുന്ന ഫംഗ്ഷനുകൾ.

താഴെ നാം ഒരു ലളിതമായ ഇൻ-মെമ്മറി മുൻഗണനാ സംഭരണശേഷി നടപ്പാക്കുന്നു (പ്രോഡക്ഷനിൽ ഇതിന് ഡാറ്റാബേസ് അല്ലെങ്കിൽ വക്തോർ ഇണ്ടക്സ് പിന്തുണ നൽകണം) അത് ഏജന്റിന് ഉപയോഗിക്കാൻ സാധിക്കുന്ന ടൂളുകളായി പ്രദർശിപ്പിക്കുന്നു.

### ഉപകടനവിന്യാസം
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ദൃശ്യാവസ്ഥ 1 — ആദ്യമായി യാത്ര ചെയ്യുന്ന ഉപഭോക്താവ് വാർഷിക യാത്ര ബുക്കുചെയ്യുന്നു

സാരാ ആദ്യമായാണ് സന്ദർശിക്കുന്നത്. ഏജന്റിന് അവളുടെ ഇഷ്ടാനുസൃതികൾ ഉപകരണങ്ങളുടെ മുഖേന സൃഷ്ടിച്ച്, അവ ഹോട്ടലുകൾ ശുപാരിശുചെയ്യാൻ ഉപയോഗിക്കണം.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### അവസ്ഥ 2 — സാറาห์ ആഴ്ചകൾക്ക് ശേഷം വീണ്ടും വരുന്നു

സാറാഹ് ഒരു **പുതിയ ത്രെഡ്** ആരംഭിക്കുന്നു (പുതിയ സെഷൻ അനുകരിക്കുന്നു). വർകിംഗ് മെമ്മറി ശൂന്യമാണ്, പക്ഷേ ദീർഘകാല ആഗ്രഹ സൂക്ഷിപ്പിൽ അവളുടെ വിവരങ്ങൾ ഇപ്പോഴും ഉണ്ടുണ്ട്. ഏജന്റ് അത് പുനരുദ്ധരിച്ച് ശിപാർശകൾ വ്യക്തിഗതമാക്കാൻ ഉപയോഗിക്കണം.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത് മൂന്നു തരം ഏജന്റ് മെമ്മറിയും അവ Microsoft Agent Framework ഉപയോഗിച്ച് എങ്ങനെ നടപ്പിലാക്കാമെന്നും ആണ്:

| മെമ്മറി തരം | MAF മാർഗ്ഗം | ആയുസ്സ് |
|---|---|---|
| **വർക്കിങ്** | `agent.create_session()` | ഒറ്റ സംഭാഷണം |
| **ഷോർട്ട്-ടേം** | ഒരു ത്രെഡ് ഉള്ളിൽ സംഭരിച്ചിട്ടുള്ള സന്ധർഭം | ഒറ്റ ജോലി / സെഷൻ |
| **ലോങ്-ടേം** | `@tool` ഫംഗ്ഷനുകൾ വഴി ലഭിക്കുന്ന ബാഹ്യ സ്റ്റോർ | സെഷനുകൾക്കുമ്പോഴെയുള്ളതിൽ |

### പ്രധാനമായ ടേക്കവേയ്സ്
1. **`agent.create_session()`** വർക്കിങ് മെമ്മറി നൽകുന്നു — ഏജന്റ് ഒരു സെഷനിൽ മുഴുവൻ സംഭാഷണ ചരിത്രവും കാണുന്നു.
2. **പുതിയ സെഷനുകൾ സന്ധർഭം നഷ്ടപ്പെടുത്തുന്നു** — ലോങ്-ടേം മെമ്മറി ഇല്ലെങ്കിൽ ഏജന്റ് മുൻ സംഭാഷണങ്ങൾ ഓർക്കാനാകില്ല.
3. **`@tool` ഫംഗ്ഷനുകൾ** ഇടവേള കവിഞ്ഞുകൊണ്ടുതന്നെ സഹായിക്കുന്നു — അവർ ഏജന്റിനെ സ്ഥിരമായ സ്റ്റോറിൽ നിന്നുള്ള വിവരങ്ങൾ സൂക്ഷിക്കാനും തിരികെക്കെടുക്കാനും അനുവദിക്കുന്നു.
4. **വ്യക്തിഗതമാക്കൽ സമയാനുസൃതമായി മെച്ചപ്പെടുന്നു** — കൂടുതൽ ഇഷ്ടാനുസരണം സൂക്ഷിക്കുമ്പോൾ ഏജന്റിന് നല്ല ശുപാർശകൾ നൽകാം.

### യാഥാർത്ഥ്യ ആർജിത അപ്ലിക്കേഷനുകൾ
- **കസ്റ്റമർ സർവീസ്**: ഉപഭോക്താവിന്റെ ചരിത്രവും ഇഷ്ടങ്ങളും ഓർമ്മിക്കുക
- **വ്യക്തിഗത അസിസ്റ്റന്റുകൾ**: ദിവസങ്ങളിലോ ആഴ്ചകളിലോ സന്ധർഭം നിലനിർത്തുക
- **ഹെൽത്ത്‌കെയർ**: രോഗിയുടെ വിവരംകളും ഇഷ്ടങ്ങളും പിന്തുടരുക
- **ഇ-കൊമേഴ്‌സ്**: ചരിത്രം അടിസ്ഥാനമാക്കിയുള്ള വ്യക്തിഗത ഷോപ്പിംഗ്

### തുടര്‍ന്നുള്ള പടികൾ
- മെമ്മറിയിലുള്ള ഡിക്ഷണറി ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ സ്റ്റോർ (ഉദാ. Azure AI Search) ഉപയോഗിച്ച് മാറ്റുക
- സമയ സെൻസിറ്റീവ് വിവരങ്ങൾക്ക് മെമ്മറി കാലഹരണപ്പെടുത്തൽ ചേർക്കുക
- പങ്കുപെടുന്ന മെമ്മറിയുള്ള മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കുക
- അറിവ്-ഗ്രാഫ് പിന്തുണയുള്ള മെമ്മറിയിനായി Cognee നോട്ട്‌ബുക്ക് പരിശോധിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
